# gnina docking — the simple version
Dock FA19 into COX-2 with AI (CNN) scoring, in 5 short cells.
**Use a GPU runtime** (Runtime → Change runtime type → GPU). gnina takes a plain PDB receptor and an SDF ligand — no format conversion needed.

**1. Install gnina**

In [ ]:
!wget -q https://github.com/gnina/gnina/releases/latest/download/gnina -O /usr/local/bin/gnina && chmod +x /usr/local/bin/gnina
!pip -q install rdkit requests 2>/dev/null
!gnina --version | head -1

**2. Receptor + box reference** — download COX-2 (3LN1), split into protein and the bound celecoxib (used to place the box)

In [ ]:
import requests
pdb = requests.get("https://files.rcsb.org/download/3LN1.pdb", timeout=60).text.splitlines()
open("rec.pdb","w").write("\n".join(l for l in pdb if l.startswith("ATOM") and l[21]=="A"))
open("ref.pdb","w").write("\n".join(l for l in pdb if l.startswith("HETATM") and l[17:20].strip()=="CEL" and l[21]=="A"))
print("receptor + reference ligand ready")

**3. Ligand** — FA19 from SMILES to a 3D SDF

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
m = Chem.AddHs(Chem.MolFromSmiles("COc1ccc(C(=O)Oc2ccc(/C=C/C(=O)O)cc2OC)cc1"))
AllChem.EmbedMolecule(m, randomSeed=42); AllChem.MMFFOptimizeMolecule(m)
Chem.MolToMolFile(m, "fa19.sdf")

**4. Dock** — box is placed automatically around the celecoxib; `--cnn_scoring rescore` turns on the AI scoring

In [ ]:
!gnina -r rec.pdb -l fa19.sdf --autobox_ligand ref.pdb -o out.sdf.gz --cnn_scoring rescore --seed 42

**5. Read the AI scores** — CNNscore = pose confidence (0–1); CNNaffinity = predicted binding strength (higher = tighter)

In [ ]:
import gzip
from rdkit import Chem
for i, mol in enumerate(Chem.ForwardSDMolSupplier(gzip.open("out.sdf.gz")), 1):
    if mol:
        print(f"pose {i}: CNNscore {float(mol.GetProp('CNNscore')):.3f} | "
              f"CNNaffinity {float(mol.GetProp('CNNaffinity')):.2f} | "
              f"affinity {float(mol.GetProp('minimizedAffinity')):.2f} kcal/mol")